# Dil modeli ortamı: Colab'da üç mod (kural / dil modeli / hibrit)

Bu defter, standart PDF'lerinden test şartlarını üç yolla çıkarıp yan yana ölçmek içindir: **yalnız kural**,
**yalnız dil modeli** (yerel çalışan `qwen3:8b`, 4 bit, düşünme kapalı) ve **hibrit** (kural öncelikli, model
tamamlar).

> **Önemli: Colab yalnız geliştirme ve ölçüm içindir.** Nihai sistem internetsiz, kendi bilgisayarında çalışır.
> Bu defterde internet yalnızca paketleri ve model dosyasını indirmek için kullanılır. Modelle konuşan kod
> (`src/malzeme_kabul/cikarim/yerel_model.py`) yalnız `localhost`'a bağlanır; Colab'da da model aynı makinede çalışan
> Ollama ya da llama.cpp sunucusundadır, hiçbir dış API çağrılmaz.
>
> Colab'a yalnızca kamuya açık standartları (data/raw altındaki MIL-PRF PDF'leri) ve üretilmiş sentetik
> sertifikaları yükle. Gerçek tedarikçi sertifikalarını ve gizli belgeleri Colab'a yükleme.

**Dil modeli yalnız okur.** Model her sayfadan test adını, metodu, sınır değerlerini, birimi ve madde numarasını
`TestSarti` şemasına zorlanmış JSON olarak döner. Birim dönüşümü ve sınır karşılaştırması kesin kurallarla
Python'da yapılır (CLAUDE.md, kural 1).

**Hibrit** (`deney/hibrit_standart.py`): kural çıktısı aynen kalır; model yalnız kuralın bulmadığı ve özgün metni
söylediği sayfada geçen testleri ekler; eklenen testin sınırını kural hesaplar; kanıtsız test atılır. Kural ile
model aynı test için farklı değer, birim ya da yön bulursa bu **anlaşmazlık** olarak yazılır ve o şart insan
incelemeli işaretlenir (`yon="belirsiz"`).

Hangi hücreleri çalıştıracaksın (sırayla):
1. GPU denetimi (en üstteki hücre), **1. Ayarlar**
2. **2. Depoyu kur** (2a, 2b, 2c)
3. **3. Ollama'yı kur ve başlat** (üç hücre)
4. İstersen **4. Tek sayfayla dene**
5. **5. Üç modu çalıştır**
6. **6. Özet tablosu**, **7. Anlaşmazlıklar ve sayfa hataları**
7. **9. Çıktıları indir**

**8. (İsteğe bağlı) llama.cpp** bölümü Ollama yerine kullanılabilir; normal akışta atla.

Çalışma zamanı: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**. T4'te 15 GB bellek var; 8B
modelin 4 bitlik (Q4_K_M) hali yaklaşık 5,2 GB tutar ve 8K bağlamla rahat sığar.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 1. Ayarlar

Tek model: `qwen3:8b`, düşünme kipi kapalı (`--dusunme-kapali`). Üç standart da geliştirme kümesindedir
(data/bolunme.yaml); test kümesindeki bir belge verilirse betik reddeder. Çıktılar `CIKTI` klasörüne yazılır.
Değişiklik `main`'e girmediyse `DAL`'a çalıştığın dalın adını yaz.

> **`DAL`'ı değiştirirsen önce `/content/malzeme-kabul-ai` klasörünü sil** (ör. `!rm -rf /content/malzeme-kabul-ai`).
> 2a hücresi klasör varsa yeniden klonlamaz; silmezsen eski dalın kodu çalışır.

In [ ]:
REPO_URL = "https://github.com/nnidaozer/malzeme-kabul-ai.git"
DAL = "main"

MODEL = "qwen3:8b"          # Ollama etiketi (varsayılan nicemleme Q4_K_M)
DUSUNME_KAPALI = True       # Qwen3'ün düşünme kipi kapatılır: --dusunme-kapali
BELGELER = [                # yalnız geliştirme kümesi (data/bolunme.yaml); test kümesi kilitli
    "data/raw/MIL-PRF-23377K.pdf",
    "data/raw/MIL-PRF-81733D.pdf",
    "data/raw/MIL-PRF-85285E.pdf",
]
BAGLAM = 8192
CIKTI = "deney/sonuclar/colab_uc_mod"

## 2. Depoyu kur

Depo gizli olduğu için klonlama bir GitHub erişim belirteci (token) ister. Belirteç Colab'ın **Gizli anahtarlar**
(Secrets, sol menüdeki 🔑) bölümünde `GITHUB_TOKEN` adıyla durur; hücre onu oradan okur. Belirteç ekrana
yazdırılmaz, deftere ya da depoya (`.git/config`) kaydedilmez. Nasıl oluşturulacağı:
`docs/dil_modeli_ortami.md` → "Colab için GitHub belirteci".

Aşağıdaki üç hücreyi sırayla çalıştır. Çalışma zamanı yeniden başlatılırsa üçünü de tekrar çalıştır.

In [ ]:
# 2a. Depoyu Secrets'taki belirteçle klonla
import base64, os, subprocess, sys

DEPO_KLASORU = "/content/malzeme-kabul-ai"

if not os.path.isdir(os.path.join(DEPO_KLASORU, ".git")):
    from google.colab import userdata
    try:
        belirtec = userdata.get("GITHUB_TOKEN")
    except userdata.SecretNotFoundError:
        raise RuntimeError("Secrets'ta GITHUB_TOKEN yok. Sol menüde 🔑 → 'Add new secret' ile ekle.") from None
    except userdata.NotebookAccessError:
        raise RuntimeError("GITHUB_TOKEN için bu deftere erişim izni verilmemiş. 🔑 bölümünde "
                           "'Notebook access' anahtarını aç.") from None
    # Belirteç adrese yazılmaz; yalnız bu komut için başlık olarak verilir, .git/config'e girmez.
    baslik = "Authorization: Basic " + base64.b64encode(f"x-access-token:{belirtec}".encode()).decode()
    sonuc = subprocess.run(
        ["git", "-c", f"http.extraHeader={baslik}", "clone", "--depth", "1", "--branch", DAL, REPO_URL, DEPO_KLASORU],
        capture_output=True, text=True,
    )
    hata = sonuc.stderr.replace(belirtec, "***")
    del belirtec, baslik
    if sonuc.returncode != 0:
        raise RuntimeError("Klonlama başarısız. Belirtecin süresi dolmuş, depo seçilmemiş ya da 'Contents: "
                           "Read-only' izni verilmemiş olabilir.\n" + hata)
    print("Depo klonlandı:", DEPO_KLASORU)
else:
    print("Depo zaten var:", DEPO_KLASORU)

os.chdir(DEPO_KLASORU)

In [ ]:
# 2b. Paketi kur
kurulum = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], capture_output=True, text=True)
if kurulum.returncode != 0:
    print(kurulum.stdout[-3000:], kurulum.stderr[-3000:])
    raise RuntimeError("pip install -e . başarısız")

# Çalışan çekirdek, pip'in yeni eklediği yolu görmez ("No module named 'malzeme_kabul'" hatasının nedeni).
# src/ klasörünü yola elle ekle; böylece çekirdeği yeniden başlatmak gerekmez.
import importlib
SRC = os.path.join(DEPO_KLASORU, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
if DEPO_KLASORU not in sys.path:  # deney/ paketi (yalnız dil modeli çıkarıcısı) depo kökünde
    sys.path.insert(0, DEPO_KLASORU)
importlib.invalidate_caches()
print("Kurulum tamam")

In [ ]:
# 2c. Kurulumu doğrula
os.chdir(DEPO_KLASORU)
import malzeme_kabul
from malzeme_kabul.cikarim.yerel_model import YerelModelIstemcisi
from deney.yalniz_dil_modeli import cikarim_semasi
from deney.uc_mod import uc_mod_calistir

assert os.path.realpath(malzeme_kabul.__file__).startswith(DEPO_KLASORU), malzeme_kabul.__file__
# Betikler ayrı süreçte çalışır; orada da paket bulunmalı.
subprocess.run([sys.executable, "-c", "import malzeme_kabul.cikarim.yerel_model"], check=True)
eksik = [pdf for pdf in BELGELER if not os.path.isfile(pdf)]
assert not eksik, f"data/raw altında standart PDF'i yok: {eksik}"
print("Paket:", os.path.dirname(malzeme_kabul.__file__))
print("Sürüm:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("Şema alanları:", list(cikarim_semasi()["$defs"]["TestSarti"]["properties"])[:5], "...")
print("✓ Kurulum doğrulandı")

## 3. Ollama'yı kur ve başlat

Sunucu yalnız `127.0.0.1:11434` adresini dinler. Model dosyaları `/content/ollama_modelleri` altına iner; oturum
kapanınca silinir. Her yeni oturumda modeli yeniden indirmek gerekir (8B için 1–2 dakika).

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import json, subprocess, time, urllib.request

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MODELS"] = "/content/ollama_modelleri"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
ollama_sunucu = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2) as y:
            print("Ollama hazır:", json.load(y)["version"])
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama başlamadı, /content/ollama.log dosyasına bak")

In [ ]:
!ollama pull {MODEL}
!ollama list

## 4. Tek sayfayla dene

Önce bir sayfayı okutup modelin şemaya uygun JSON döndüğünü gör. Aynı istek bütün belgede sayfa sayfa yapılır.

In [ ]:
import time

from malzeme_kabul.cikarim.kural_cikarici import metni_normallestir
from malzeme_kabul.belge.pdf_okuyucu import pdf_oku
from malzeme_kabul.cikarim.yerel_model import YerelModelIstemcisi
from deney.yalniz_dil_modeli import cikarim_semasi, istem_olustur, yaniti_coz

belge = pdf_oku("data/raw/MIL-PRF-23377K.pdf")
SAYFA = 6   # deneme sayfası; istediğin sayfayı yaz
istemci = YerelModelIstemcisi(model=MODEL, baglam=BAGLAM, dusunme=False if DUSUNME_KAPALI else None)
mesajlar = istem_olustur(metni_normallestir(belge.sayfalar[SAYFA - 1].metin), SAYFA, "MIL-PRF-23377", "K")

t0 = time.perf_counter()
ham = istemci.sohbet(mesajlar, cikarim_semasi())
print(f"{time.perf_counter() - t0:.1f} s")
testler, hatalar = yaniti_coz(ham)
for t in testler:
    print(t.kaynak.madde, "|", t.test_adi, "|", t.sinir.alt, t.sinir.ust, t.sinir.birim, "|", sorted(t.test_turleri))
print("Hatalar:", hatalar)

## 5. Üç modu çalıştır

Her belge için `deney/uc_mod.py` çağrılır: kural, yalnız dil modeli ve hibrit çıktıları ile rapor
`CIKTI` altına yazılır (`<BELGE>_kural.json`, `<BELGE>_dil_modeli.json`, `<BELGE>_hibrit.json`,
`<BELGE>_uc_mod_rapor.json`, gold varsa `<BELGE>_uc_mod_olcum.csv`). Gold olarak `data/gold_taslak` ve
`data/gold_codex` altındaki taslaklar (varsa) yalnız okunur.

**Colab koparsa:** 2a–2c, 3. bölüm ve 1. Ayarlar hücrelerini yeniden çalıştırıp bu hücreyi yeniden çalıştırmak
yeterli. Dil modeli her sayfanın sonucunu `CIKTI/<BELGE>_ara/` altına ara kayıt olarak yazar; kaydı olan sayfalar
modele yeniden gönderilmez, kaldığı yerden devam eder. (Oturum tamamen sıfırlandıysa `/content` silinmiştir; o
durumda baştan başlar. Ara kayıtları korumak için CIKTI klasörünü ara ara 9. bölümle indir.)

**Tahmini süre** (T4, `qwen3:8b`, düşünme kapalı; MIL-PRF-23377K'da ölçülen ~36 s/sayfa üzerinden kaba tahmin):

| Belge | Sayfa | Süre |
|---|---|---|
| MIL-PRF-23377K | 19 | ~11 dk |
| MIL-PRF-81733D | 34 | ~20 dk |
| MIL-PRF-85285E | 24 | ~14 dk |
| **Toplam** | 77 | **~45 dk** |

Buna kurulum ve model indirme için ~10 dk ekle. Kural modu ve hibrit birleştirme saniyeler sürer.

Model ilk çağrıda belleğe yüklenir; bu süre ölçüme karışmasın diye önce bir ısınma isteği gönderilir.

In [ ]:
import subprocess, sys
from malzeme_kabul.cikarim.yerel_model import YerelModelIstemcisi

# Isınma: modeli belleğe yükle, süreye katılmasın.
YerelModelIstemcisi(model=MODEL, baglam=BAGLAM, en_cok_token=16, dusunme=False if DUSUNME_KAPALI else None,
                    ).sohbet([{"role": "user", "content": "ok"}], {"type": "object"})

ek = ["--dusunme-kapali"] if DUSUNME_KAPALI else []
basarisiz = []
for pdf in BELGELER:
    print("###", pdf)
    sonuc = subprocess.run([sys.executable, "deney/uc_mod.py", pdf, "--model", MODEL, "--baglam", str(BAGLAM),
                            "--cikti", CIKTI, *ek])
    if sonuc.returncode != 0:
        basarisiz.append(pdf)
if basarisiz:
    raise RuntimeError(f"Çalışmayan belgeler (yukarıdaki hataya bak, hücreyi yeniden çalıştır): {basarisiz}")
print("Üç mod tamam ->", CIKTI)

## 6. Özet tablosu

Satırlar belge × mod: test sayısı, süre, insan incelemeli sınır sayısı (yönü `belirsiz` olan sayısal sınırlar),
kural–model anlaşmazlığı (belge düzeyinde; yalnız hibrit satırında dolu, çünkü anlaşmazlığı hibrit insan incelemesine gönderir) ve her gold için GENEL (mikro)
precision / recall / F1.

**Güven skoru için birincil sayı "anlaşmazlıklı test" sütunudur** (kaç testte kural ile model ayrıştı). "anlaşmazlık (kayıt)" aynı testteki ayrı alan farklarını da sayar; ikincil bilgidir. Ölçüm hizalıdır: gold'daki kayıtla adı benzer kayıt aynı kayıt sayılır.

> **Sonuçlar geçicidir.** `data/gold_taslak` ve `data/gold_codex` henüz proje sahibinin onayladığı cevap
> anahtarları değildir; onaylanıp `data/gold/` altına taşınana kadar bu sayılar yalnız ön bilgidir. **İstemi ya
> da çıkarıcıyı bu sonuçlara bakarak ayarlama**; yoksa sistem cevap anahtarına göre ayarlanmış olur.

In [ ]:
import glob, json, os
import pandas as pd

def ozet_tablosu(klasor=CIKTI):
    satirlar = []
    for yol in sorted(glob.glob(os.path.join(klasor, "*_uc_mod_rapor.json"))):
        with open(yol, encoding="utf-8") as f:
            r = json.load(f)
        olcum = {}
        csv_yolu = yol.replace("_uc_mod_rapor.json", "_uc_mod_olcum.csv")
        if os.path.isfile(csv_yolu):
            tablo = pd.read_csv(csv_yolu)
            for s in tablo[tablo["alan"] == "GENEL (mikro)"].itertuples():
                olcum[(s.gold, s.mod)] = (s.precision, s.recall, s.f1)
        for mod, m in r["modlar"].items():
            satir = {
                "belge": r["belge"], "mod": mod, "test": m["test_sayisi"], "süre (s)": m["sure_s"],
                "insan_incelemeli_sinir": m["insan_incelemeli_sinir"],
                # Belge düzeyi: yalnız hibrit satırında dolu. Güven skoru için birincil sayı "anlaşmazlıklı test".
                "anlaşmazlıklı test": r["anlasmazlikli_test_sayisi"] if mod == "hibrit" else None,
                "anlaşmazlık (kayıt)": r["anlasmazlik_sayisi"] if mod == "hibrit" else None,
            }
            for gold in sorted({g for g, _ in olcum}):
                p, rc, f1 = olcum.get((gold, mod), (None, None, None))
                satir.update({f"{gold} P": p, f"{gold} R": rc, f"{gold} F1": f1})
            satirlar.append(satir)
    return pd.DataFrame(satirlar).fillna("")

ozet_tablosu()

## 7. Anlaşmazlıklar ve sayfa hataları

Kural ile modelin aynı test için farklı bulduğu alanlar (kriter türü, alt, üst, birim, yön), hibrite eklenmeyen
kanıtsız dil modeli testleri ve modelin hata verdiği sayfalar.

In [ ]:
import glob, json, os

for yol in sorted(glob.glob(os.path.join(CIKTI, "*_uc_mod_rapor.json"))):
    with open(yol, encoding="utf-8") as f:
        r = json.load(f)
    h, dm = r["hibrit_raporu"], r["dil_modeli_raporu"]
    print(f"== {r['belge']}: {h['anlasmazlik_sayisi']} anlaşmazlık ({h['anlasmazlikli_test_sayisi']} test), "
          f"eklenen {len(h['eklenen'])}, kanıtsız {len(h['kanitsiz'])}, "
          f"devam edilen sayfa {len(dm['devam_edilen_sayfalar'])}")
    for k in h["anlasmazliklar"]:
        print(f"  {k['test_id']} ({k['model_id']}) {k['alan']}: kural={k['kural']} model={k['model']}")
    if h["isaretlenemeyen"]:
        print("  işaretlenemeyen (sayısal değil):", ", ".join(h["isaretlenemeyen"]))
    for s in dm["sayfalar"]:
        for hata in s["hatalar"]:
            print("  sayfa", s["sayfa"], "|", hata)

## 8. (İsteğe bağlı) llama.cpp ile aynı iş

Ollama yerine llama.cpp'nin `llama-server` sunucusu da kullanılabilir. Şema `response_format` içinde
`json_schema` olarak gider ve sunucu onu dilbilgisine (grammar) çevirerek üretimi zorlar. Derleme T4'te 5–10
dakika sürer. Ollama sunucusu çalışıyorsa bellek için önce onu durdur. Aynı model (Qwen3-8B, Q4_K_M) kullanılır;
çıktılar ayrı klasöre (`CIKTI + "_llamacpp"`) yazılır. Not: düşünme kipini kapatma isteği yalnız Ollama'ya
gönderilir.

In [ ]:
if "ollama_sunucu" in globals():
    ollama_sunucu.terminate()
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --config Release -j 4 --target llama-server 2>&1 | tail -2

In [ ]:
import json, subprocess, time, urllib.request
from huggingface_hub import hf_hub_download

# Ollama'daki modelin GGUF karşılığı (Q4_K_M).
gguf_yolu = hf_hub_download("Qwen/Qwen3-8B-GGUF", "Qwen3-8B-Q4_K_M.gguf", local_dir="/content/gguf")

llama_sunucu = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server", "-m", gguf_yolu, "-ngl", "99", "-c", str(BAGLAM),
     "--host", "127.0.0.1", "--port", "8080"],
    stdout=open("/content/llama_server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2) as y:
            if json.load(y).get("status") == "ok":
                print("llama-server hazır")
                break
    except OSError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("llama-server başlamadı, /content/llama_server.log dosyasına bak")

In [ ]:
import subprocess, sys

for pdf in BELGELER:
    subprocess.run([sys.executable, "deney/uc_mod.py", pdf, "--model", MODEL, "--arka-uc", "llamacpp",
                    "--baglam", str(BAGLAM), "--cikti", CIKTI + "_llamacpp"], check=True)
ozet_tablosu(CIKTI + "_llamacpp")

## 9. Çıktıları indir

Colab oturumu kapanınca dosyalar silinir. `CIKTI` klasörünü (ara kayıtlar dahil; varsa llama.cpp çıktıları da)
zip olarak indir. Depoya eklemek istersen ayrı bir PR ile `deney/sonuclar/colab_uc_mod/` altına koy;
`data/gold/` klasörüne hiçbir şey yazılmaz.

In [ ]:
!cd /content/malzeme-kabul-ai && zip -qr /content/colab_uc_mod.zip {CIKTI}*
from google.colab import files
files.download("/content/colab_uc_mod.zip")